## TECH CHALLENGE - FASE 4

## Análise Multimodal para Detecção de Riscos em Saúde da Mulher

Este notebook apresenta a implementação inicial de um pipeline multimodal capaz de analisar dados de vídeo e áudio com o objetivo de identificar possíveis sinais de risco clínico e psicológico.

A solução é uma evolução do assistente médico desenvolvido na fase anterior, incorporando análise de comportamento (vídeo) e linguagem (áudio), permitindo maior capacidade de detecção precoce de situações críticas.

## Objetivo

Desenvolver um pipeline multimodal que:

- Analise vídeos para identificar padrões comportamentais e sinais de desconforto
- Processe áudios de consultas para detectar sinais emocionais e psicológicos
- Combine os resultados das diferentes modalidades
- Gere um alerta baseado no nível de risco identificado

Este sistema atua como suporte à triagem clínica, não realizando diagnósticos médicos.

## Importação de Bibliotecas e Módulos

Nesta etapa, são importadas as funções desenvolvidas no projeto para análise de vídeo, áudio e fusão multimodal.

In [1]:
import sys
sys.path.append("..")

In [2]:
from src.multimodal.video_processor import analyze_video
from src.multimodal.audio_processor import analyze_audio
from src.multimodal.multimodal_fusion import calculate_multimodal_risk
from src.multimodal.alert_generator import generate_alert

## Configuração do Ambiente

Configuração de caminhos e preparação do ambiente para execução das análises.

In [3]:
from pathlib import Path

print(Path.cwd())

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks


In [4]:
video_result = analyze_video("../data/multimodal/video/sample.mp4")
audio_result = analyze_audio("../data/multimodal/audio/sample.wav")

risk_result = calculate_multimodal_risk(video_result, audio_result)
alert = generate_alert(risk_result)

video_result, audio_result, risk_result, alert

({'modality': 'video',
  'risk_score': 0.55,
  'risk_level': 'medio',
  'flags': ['movimento corporal identificado',
   'análise visual inicial executada'],
  'metadata': {'video_path': '..\\data\\multimodal\\video\\sample.mp4',
   'fps': 0,
   'frame_count': 0,
   'duration_seconds': 0,
   'status': 'mock_video_file'}},
 {'modality': 'audio',
  'risk_score': 0.3,
  'risk_level': 'medio',
  'flags': [],
  'transcription': 'Não foi possível transcrever o áudio.'},
 {'final_score': 0.4,
  'risk_level': 'medio',
  'alert': False,
  'evidences': ['movimento corporal identificado',
   'análise visual inicial executada'],
  'video_score': 0.55,
  'audio_score': 0.3},
 'Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.')

## Carregamento de Dados de Entrada

Nesta fase inicial, são utilizados dados simulados (mock) para representar arquivos de vídeo e áudio.
Em etapas futuras, estes dados serão substituídos por arquivos reais e integrados a serviços como Azure Speech e modelos de visão computacional.

## Análise de Vídeo

O módulo de análise de vídeo tem como objetivo identificar padrões visuais que possam indicar:

- Desconforto físico
- Movimentos anormais
- Postura corporal inadequada

Nesta etapa inicial, a análise é simulada e será futuramente substituída por modelos de visão computacional (YOLOv8).

## Análise de Áudio

O módulo de análise de áudio processa informações da fala da paciente, buscando identificar:

- Sinais de ansiedade
- Indícios de medo
- Relatos de sintomas relevantes

A análise utiliza transcrição simulada, sendo posteriormente integrada com serviços como Azure Speech-to-Text.

In [5]:
import sys
sys.path.append("..")

from src.workflows.langgraph_flow import build_medical_assistant_graph
from src.llm.ollama_client import get_llm
from src.rag.vector_store import load_vector_store
from src.rag.retriever import get_retriever
from src.rag.retriever import retrieve_context

In [6]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store


## Execução do Fluxo Multimodal com LangGraph

Nesta etapa, o pipeline multimodal é executado por meio do LangGraph, mantendo a arquitetura evolutiva desenvolvida na fase anterior.

O fluxo recebe como entrada uma pergunta clínica, um identificador opcional de paciente, um arquivo de vídeo e um arquivo de áudio. Em seguida, o sistema executa os nós de guardrails, recuperação de contexto via RAG, análise de vídeo, análise de áudio, fusão multimodal, geração de resposta e registro da interação.

Nesta execução inicial, os dados de áudio e vídeo ainda são simulados, permitindo validar a integração entre os módulos antes da substituição por modelos reais, como YOLOv8 para vídeo, Azure Speech-to-Text para transcrição e GPT para interpretação textual especializada.

In [7]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/sample.wav"
}

result = app.invoke(state)
result

{'question': 'Avalie possíveis sinais de risco no atendimento da paciente.',
 'risk_level': 'low',
 'action': 'allow',
 'reason': 'Pergunta informativa.',
 'docs': [Document(id='b0801d89-44f4-4f6c-8a1e-b74b7a03c78a', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='fe392124-22dc-406d-a1cb-da95babe86e9', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='a29fa53d-b57a-47d2-a544-780df6879c83', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia')],
 'context': 'Recurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia',
 'patient_id': None,
 'patient_context': 'No patien

## Interpretação do Resultado

O resultado demonstra que o fluxo multimodal foi executado com sucesso.

A saída contém:

- A classificação de risco inicial realizada pelos guardrails
- Os documentos recuperados pela base vetorial FAISS
- O contexto estruturado do paciente, quando disponível
- O contexto multimodal gerado a partir das análises de vídeo e áudio
- O score final de risco multimodal
- A resposta gerada pelo assistente médico
- As fontes utilizadas para rastreabilidade
- O status final da execução

Nesta primeira versão, o risco final foi classificado como médio, com evidências simuladas relacionadas a movimento corporal, medo, cansaço e dificuldade para dormir. O alerta crítico não foi acionado, mas o sistema recomenda acompanhamento regular pela equipe de saúde.

Essa etapa confirma que a arquitetura da Fase 3 foi expandida para suportar dados multimodais, mantendo os princípios de segurança, explicabilidade e rastreabilidade.

In [8]:
print(result["answer"])
print(result["multimodal_result"])

 Based on the provided structured patient data, multimodal analysis, and evidence from the initial visual assessment and identified body movement, there is currently no critical alert detected. The final multimodal score of 0.4 indicates a medium risk level, and no alarm has been generated. Therefore, it's recommended that the patient continues with regular follow-ups by her healthcare team for ongoing monitoring.

However, it's important to note that the absence of critical alerts does not necessarily mean the absence of potential risks or symptoms. The recurrent mention of Recurrent Adult Acute Lymphoblastic Leukemia in the medical knowledge retrieved could be a relevant factor for further investigation during the patient's regular follow-ups.

For a definitive evaluation and management plan, please consult with the healthcare team that is familiar with the patient's case and medical history.
{'final_score': 0.4, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corpora

In [9]:
print(result["multimodal_context"])

MULTIMODAL ANALYSIS:
Video risk score: 0.55
Audio risk score: 0.3
Final multimodal score: 0.4
Risk level: medio
Alert generated: False

Evidence:
- movimento corporal identificado
- análise visual inicial executada

Alert message:
Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.


In [11]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app1 = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

Base vetorial carregada de: ../data/vector_store


In [12]:
state = {
    "question": "Avalie possíveis sinais de câncer de mama.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/sample.wav"
}

result1 = app1.invoke(state)
result1

{'question': 'Avalie possíveis sinais de câncer de mama.',
 'risk_level': 'low',
 'action': 'allow',
 'reason': 'Pergunta informativa.',
 'docs': [Document(id='b0801d89-44f4-4f6c-8a1e-b74b7a03c78a', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='fe392124-22dc-406d-a1cb-da95babe86e9', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='a29fa53d-b57a-47d2-a544-780df6879c83', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia')],
 'context': 'Recurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia',
 'patient_id': None,
 'patient_context': 'No patient data was provide

In [13]:
print(result1["answer"])
print(result1["multimodal_result"])

 Based on the provided context, there are no signs or symptoms of breast cancer detected in this case. The multimodal analysis indicates a medium risk level with no critical alert. The identified symptoms include movement observed, initial visual analysis, fear, fatigue, and difficulty sleeping. These symptoms could be associated with various conditions such as stress, anxiety, or other non-cancerous health issues. However, without specific patient data, it is not possible to definitively assess the possibility of breast cancer. It's recommended that the patient follows regular check-ups with their healthcare team for thorough evaluation and appropriate care.
{'final_score': 0.67, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada', 'medo', 'cansaço', 'dificuldade para dormir'], 'video_score': 0.55, 'audio_score': 0.75}


In [14]:
print(result1["multimodal_context"])

MULTIMODAL ANALYSIS:
Video risk score: 0.55
Audio risk score: 0.75
Final multimodal score: 0.67
Risk level: medio
Alert generated: False

Evidence:
- movimento corporal identificado
- análise visual inicial executada
- medo
- cansaço
- dificuldade para dormir

Alert message:
Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.


## Análise do Resultado – Limitações da Abordagem Multimodal

Neste experimento, foi realizada uma pergunta sobre possíveis sinais de câncer de mama, enquanto o pipeline multimodal utilizou dados comportamentais e emocionais simulados (vídeo e áudio).

O sistema respondeu corretamente ao indicar que não é possível identificar câncer de mama com base apenas em sinais comportamentais ou emocionais, reforçando a necessidade de exames clínicos e dados médicos específicos.

A análise multimodal contribuiu apenas com um score de risco geral (médio), baseado em sinais como medo, cansaço e dificuldade para dormir, que são inespecíficos e podem estar associados a diversas condições não relacionadas ao câncer.

Esse resultado evidencia uma limitação importante da abordagem:
dados multimodais como áudio e vídeo são úteis para triagem e identificação de bem-estar psicológico, mas não são suficientes para diagnóstico de doenças específicas como câncer.

O sistema manteve comportamento seguro, evitando conclusões médicas indevidas e recomendando acompanhamento profissional, alinhado aos princípios de segurança definidos na arquitetura.

In [10]:
import speech_recognition as sr

print("OK")

OK


In [11]:
import speech_recognition as sr

def transcribe_audio(audio_path):
    recognizer = sr.Recognizer()

    with sr.AudioFile(audio_path) as source:
        audio = recognizer.record(source)

    text = recognizer.recognize_google(audio, language="pt-BR")

    return text


audio_path = "../data/multimodal/audio/audio1.wav"

transcription = transcribe_audio(audio_path)
print(transcription)

audio_path2 = "../data/multimodal/audio/audio2.wav"

transcription2 = transcribe_audio(audio_path2)
print(transcription2)

audio_path3 = "../data/multimodal/audio/audio3.wav"

transcription3 = transcribe_audio(audio_path3)
print(transcription3)

Oi boa tarde venho apenas para uma consulta de rotina e gostaria de fazer um check-up geral para avaliar como está minha saúde
eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido
eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada


In [12]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

Base vetorial carregada de: ../data/vector_store


In [13]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/audio1.wav"
}

In [14]:
result = app.invoke(state)

print(result["audio_result"]["transcription"])
print(result["audio_result"]["flags"])
print(result["multimodal_result"])

Oi boa tarde venho apenas para uma consulta de rotina e gostaria de fazer um check-up geral para avaliar como está minha saúde
[]
{'final_score': 0.4, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada'], 'video_score': 0.55, 'audio_score': 0.3}
